# FD001 MLP Experiments

Objetivo: mejorar una MLP tabular con features temporales resumidas. No se usan CNN, LSTM, GRU, Transformers ni ventanas crudas como secuencia.


In [1]:
from collections import OrderedDict
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
for path in [RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
DATA_DIR = PROJECT_ROOT / "CMAPSSData"
EVAL_SIZE = 0.2
CUT_RULS = (20, 50, 80, 110, 140)
DEFAULT_WINDOW_SIZE = 30
DEFAULT_RUL_CAP = 125

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)


In [2]:
from src.preprocessed_FD001 import (
    prepare_fd001_temporal_validation_only,
    preprocessing_summary,
)
from src.fd001_modeling import (
    metrics_by_model,
    metrics_by_rul_bin,
    plot_validation_diagnostics,
    prediction_frame,
    regression_metrics,
    set_global_seed,
)


def add_bin_metric_columns(metrics, bin_metrics):
    result = metrics.copy()
    for label in ["0-30", "30-60", "60-90", "90+"]:
        safe_label = label.replace("-", "_").replace("+", "plus")
        subset = bin_metrics.loc[bin_metrics["rul_bin"].astype(str) == label]
        for _, row in subset.iterrows():
            mask = (
                (result["representation"] == row["representation"])
                & (result["model_name"] == row["model_name"])
            )
            result.loc[mask, f"mae_rul_{safe_label}"] = row["mae"]
            result.loc[mask, f"dangerous_error_pct_rul_{safe_label}"] = row["dangerous_error_pct"]
    return result


def available_tabular_factories(random_state=42):
    from sklearn.ensemble import (
        ExtraTreesRegressor,
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )

    factories = OrderedDict()
    availability = []
    has_external_boosting = False

    factories["RandomForestRegressor"] = lambda: RandomForestRegressor(
        n_estimators=250,
        max_depth=14,
        min_samples_leaf=3,
        random_state=random_state,
        n_jobs=-1,
    )

    try:
        from xgboost import XGBRegressor
        factories["XGBRegressor"] = lambda: XGBRegressor(
            n_estimators=160,
            max_depth=3,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=-1,
            tree_method="hist",
            verbosity=0,
        )
        has_external_boosting = True
        availability.append("XGBoost disponible: se incluye XGBRegressor.")
    except Exception as exc:
        availability.append(f"XGBoost no disponible: {type(exc).__name__}.")

    try:
        from lightgbm import LGBMRegressor
        factories["LGBMRegressor"] = lambda: LGBMRegressor(
            n_estimators=220,
            max_depth=-1,
            num_leaves=31,
            learning_rate=0.035,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=0.1,
            random_state=random_state,
            n_jobs=-1,
            verbose=-1,
        )
        has_external_boosting = True
        availability.append("LightGBM disponible: se incluye LGBMRegressor.")
    except Exception as exc:
        availability.append(f"LightGBM no disponible: {type(exc).__name__}.")

    if not has_external_boosting:
        availability.append("Sin XGBoost/LightGBM: se usan fallbacks sklearn.")
        factories["HistGradientBoostingRegressor"] = lambda: HistGradientBoostingRegressor(
            max_iter=180,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=random_state,
        )
        factories["GradientBoostingRegressor"] = lambda: GradientBoostingRegressor(
            n_estimators=160,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.85,
            random_state=random_state,
        )
        factories["ExtraTreesRegressor"] = lambda: ExtraTreesRegressor(
            n_estimators=220,
            max_depth=16,
            min_samples_leaf=2,
            random_state=random_state,
            n_jobs=-1,
        )
    return factories, availability

def fit_predict_model(prepared, model_name, model, representation, sample_weight=None):
    if sample_weight is None:
        model.fit(prepared["X_train"], prepared["y_train"])
    else:
        model.fit(prepared["X_train"], prepared["y_train"], sample_weight=sample_weight)
    preds = model.predict(prepared["X_eval"])
    return prediction_frame(
        prepared["eval_df"],
        preds,
        model_name=model_name,
        representation=representation,
    ), model


In [3]:
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

class FlexibleMLPRegressor:
    def __init__(self, hidden_layers, dropout=0.1, lr=1e-3, weight_decay=1e-4, batch_size=256,
                 max_epochs=250, patience=25, validation_fraction=0.15, random_state=42, device=None):
        self.hidden_layers = list(hidden_layers)
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.patience = patience
        self.validation_fraction = validation_fraction
        self.random_state = random_state
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model_ = None
        self.history_ = []

    def _build_model(self, input_dim):
        layers = []
        prev = input_dim
        for width in self.hidden_layers:
            layers.append(nn.Linear(prev, int(width)))
            layers.append(nn.ReLU())
            if self.dropout > 0:
                layers.append(nn.Dropout(self.dropout))
            prev = int(width)
        layers.append(nn.Linear(prev, 1))
        return nn.Sequential(*layers)

    def fit(self, X, y):
        set_global_seed(self.random_state)
        X_np = np.asarray(X, dtype=np.float32)
        y_np = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        idx = np.arange(len(X_np))
        train_idx, val_idx = train_test_split(idx, test_size=self.validation_fraction, random_state=self.random_state, shuffle=True)
        X_train = torch.tensor(X_np[train_idx], dtype=torch.float32)
        y_train = torch.tensor(y_np[train_idx], dtype=torch.float32)
        X_val = torch.tensor(X_np[val_idx], dtype=torch.float32).to(self.device)
        y_val = torch.tensor(y_np[val_idx], dtype=torch.float32).to(self.device)
        loader = DataLoader(
            TensorDataset(X_train, y_train),
            batch_size=self.batch_size,
            shuffle=True,
            generator=torch.Generator().manual_seed(self.random_state),
        )
        self.model_ = self._build_model(X_np.shape[1]).to(self.device)
        optimizer = torch.optim.Adam(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        loss_fn = nn.MSELoss()
        best_loss = np.inf
        best_state = None
        patience_left = self.patience
        self.history_ = []
        for epoch in range(1, self.max_epochs + 1):
            self.model_.train()
            train_losses = []
            for xb, yb in loader:
                xb = xb.to(self.device)
                yb = yb.to(self.device)
                optimizer.zero_grad()
                loss = loss_fn(self.model_(xb), yb)
                loss.backward()
                optimizer.step()
                train_losses.append(float(loss.detach().cpu()))
            self.model_.eval()
            with torch.no_grad():
                val_loss = float(loss_fn(self.model_(X_val), y_val).detach().cpu())
            self.history_.append({"epoch": epoch, "train_loss": float(np.mean(train_losses)), "val_loss": val_loss})
            if val_loss < best_loss - 1e-5:
                best_loss = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in self.model_.state_dict().items()}
                patience_left = self.patience
            else:
                patience_left -= 1
            if patience_left <= 0:
                break
        if best_state is not None:
            self.model_.load_state_dict(best_state)
        return self

    def predict(self, X):
        if self.model_ is None:
            raise RuntimeError("La MLP debe entrenarse antes de predecir.")
        X_np = np.asarray(X, dtype=np.float32)
        self.model_.eval()
        with torch.no_grad():
            preds = self.model_(torch.tensor(X_np, dtype=torch.float32).to(self.device))
        return np.clip(preds.detach().cpu().numpy().reshape(-1), 0.0, None)


## Configuraciones

Se prueban pocas combinaciones representativas, no una grilla completa.


In [4]:
MLP_CONFIGS = [
    {"hidden_layers": [64, 32], "dropout": 0.0, "weight_decay": 1e-4, "lr": 1e-3, "batch_size": 256},
    {"hidden_layers": [64, 32], "dropout": 0.1, "weight_decay": 1e-4, "lr": 1e-3, "batch_size": 128},
    {"hidden_layers": [128, 64, 32], "dropout": 0.1, "weight_decay": 1e-4, "lr": 1e-3, "batch_size": 256},
    {"hidden_layers": [128, 64, 32], "dropout": 0.2, "weight_decay": 1e-4, "lr": 5e-4, "batch_size": 256},
    {"hidden_layers": [256, 128, 64], "dropout": 0.1, "weight_decay": 1e-5, "lr": 1e-3, "batch_size": 256},
    {"hidden_layers": [256, 128, 64], "dropout": 0.2, "weight_decay": 1e-4, "lr": 5e-4, "batch_size": 128},
    {"hidden_layers": [128, 64, 32], "dropout": 0.0, "weight_decay": 1e-3, "lr": 1e-3, "batch_size": 256},
    {"hidden_layers": [64, 32], "dropout": 0.2, "weight_decay": 1e-3, "lr": 5e-4, "batch_size": 128},
]
MLP_CONFIGS


[{'hidden_layers': [64, 32],
  'dropout': 0.0,
  'weight_decay': 0.0001,
  'lr': 0.001,
  'batch_size': 256},
 {'hidden_layers': [64, 32],
  'dropout': 0.1,
  'weight_decay': 0.0001,
  'lr': 0.001,
  'batch_size': 128},
 {'hidden_layers': [128, 64, 32],
  'dropout': 0.1,
  'weight_decay': 0.0001,
  'lr': 0.001,
  'batch_size': 256},
 {'hidden_layers': [128, 64, 32],
  'dropout': 0.2,
  'weight_decay': 0.0001,
  'lr': 0.0005,
  'batch_size': 256},
 {'hidden_layers': [256, 128, 64],
  'dropout': 0.1,
  'weight_decay': 1e-05,
  'lr': 0.001,
  'batch_size': 256},
 {'hidden_layers': [256, 128, 64],
  'dropout': 0.2,
  'weight_decay': 0.0001,
  'lr': 0.0005,
  'batch_size': 128},
 {'hidden_layers': [128, 64, 32],
  'dropout': 0.0,
  'weight_decay': 0.001,
  'lr': 0.001,
  'batch_size': 256},
 {'hidden_layers': [64, 32],
  'dropout': 0.2,
  'weight_decay': 0.001,
  'lr': 0.0005,
  'batch_size': 128}]

In [5]:
set_global_seed(SEED)
prepared = prepare_fd001_temporal_validation_only(
    data_dir=DATA_DIR,
    eval_size=EVAL_SIZE,
    random_state=SEED,
    max_rul=DEFAULT_RUL_CAP,
    cut_ruls=CUT_RULS,
    window_size=DEFAULT_WINDOW_SIZE,
)
preprocessing_summary(prepared)


,split,rows,units,features,target_mean,target_min,target_max
0,train,16561,80,119,86.958819,0.0,125.0
1,eval,99,20,119,79.393939,20.0,140.0


## Entrenamiento MLP


In [6]:
mlp_predictions = []
mlp_config_rows = []
representation = f"temporal_w{DEFAULT_WINDOW_SIZE}"

for idx, cfg in enumerate(MLP_CONFIGS, start=1):
    model_name = f"MLP_tabular_cfg_{idx:02d}"
    print(f"Entrenando {model_name}: {cfg}")
    model = FlexibleMLPRegressor(**cfg, random_state=SEED)
    model.fit(prepared["X_train"], prepared["y_train"])
    preds = model.predict(prepared["X_eval"])
    pred_df = prediction_frame(prepared["eval_df"], preds, model_name=model_name, representation=representation)
    mlp_predictions.append(pred_df)
    mlp_config_rows.append({"model_name": model_name, **cfg, "epochs_run": len(model.history_)})

mlp_predictions = pd.concat(mlp_predictions, ignore_index=True)
mlp_metrics = metrics_by_model(mlp_predictions)
mlp_bin_metrics = metrics_by_rul_bin(mlp_predictions)
mlp_metrics = add_bin_metric_columns(mlp_metrics, mlp_bin_metrics)
config_table = pd.DataFrame(mlp_config_rows)
mlp_metrics = mlp_metrics.merge(config_table, on="model_name", how="left")
mlp_metrics = mlp_metrics.sort_values(["rmse", "mae"]).reset_index(drop=True)
mlp_metrics.head(10)


Entrenando MLP_tabular_cfg_01: {'hidden_layers': [64, 32], 'dropout': 0.0, 'weight_decay': 0.0001, 'lr': 0.001, 'batch_size': 256}
Entrenando MLP_tabular_cfg_02: {'hidden_layers': [64, 32], 'dropout': 0.1, 'weight_decay': 0.0001, 'lr': 0.001, 'batch_size': 128}
Entrenando MLP_tabular_cfg_03: {'hidden_layers': [128, 64, 32], 'dropout': 0.1, 'weight_decay': 0.0001, 'lr': 0.001, 'batch_size': 256}
Entrenando MLP_tabular_cfg_04: {'hidden_layers': [128, 64, 32], 'dropout': 0.2, 'weight_decay': 0.0001, 'lr': 0.0005, 'batch_size': 256}
Entrenando MLP_tabular_cfg_05: {'hidden_layers': [256, 128, 64], 'dropout': 0.1, 'weight_decay': 1e-05, 'lr': 0.001, 'batch_size': 256}
Entrenando MLP_tabular_cfg_06: {'hidden_layers': [256, 128, 64], 'dropout': 0.2, 'weight_decay': 0.0001, 'lr': 0.0005, 'batch_size': 128}
Entrenando MLP_tabular_cfg_07: {'hidden_layers': [128, 64, 32], 'dropout': 0.0, 'weight_decay': 0.001, 'lr': 0.001, 'batch_size': 256}
Entrenando MLP_tabular_cfg_08: {'hidden_layers': [64, 32

,representation,model_name,n_eval,mae,rmse,r2,cmapss_score,cmapss_score_mean,dangerous_error_pct,mae_rul_0_30,dangerous_error_pct_rul_0_30,mae_rul_30_60,dangerous_error_pct_rul_30_60,mae_rul_60_90,dangerous_error_pct_rul_60_90,mae_rul_90plus,dangerous_error_pct_rul_90plus,hidden_layers,dropout,weight_decay,lr,batch_size,epochs_run
0,temporal_w30,MLP_tabular_cfg_03,99,12.626827,17.051460,0.836789,449.529187,4.540699,10.101010,4.246185,0.0,8.448703,15.0,15.369181,35.0,17.660883,0.000000,"[128, 64, 32]",0.1,0.00010,0.0010,256,250
1,temporal_w30,MLP_tabular_cfg_04,99,13.523962,17.588616,0.826344,444.876062,4.493698,9.090909,4.744157,0.0,9.647462,15.0,15.674224,30.0,18.911677,0.000000,"[128, 64, 32]",0.2,0.00010,0.0005,256,250
2,temporal_w30,MLP_tabular_cfg_05,99,13.723869,17.751059,0.823122,509.341935,5.144868,10.101010,4.725164,0.0,9.677332,15.0,17.584081,35.0,18.434141,0.000000,"[256, 128, 64]",0.1,0.00001,0.0010,256,243
3,temporal_w30,MLP_tabular_cfg_06,99,14.250633,18.486643,0.808159,521.062203,5.263255,13.131313,4.289032,0.0,9.465314,15.0,18.129801,50.0,19.823838,0.000000,"[256, 128, 64]",0.2,0.00010,0.0005,128,250
4,temporal_w30,MLP_tabular_cfg_08,99,14.818981,19.361553,0.789571,715.295973,7.225212,13.131313,4.601485,0.0,10.816306,15.0,18.014331,45.0,20.472735,2.564103,"[64, 32]",0.2,0.00100,0.0005,128,250
5,temporal_w30,MLP_tabular_cfg_07,99,15.320591,20.009832,0.775243,753.779214,7.613931,11.111111,4.717534,0.0,13.684664,20.0,17.720458,30.0,20.366292,2.564103,"[128, 64, 32]",0.0,0.00100,0.0010,256,213
6,temporal_w30,MLP_tabular_cfg_02,99,15.588494,20.218365,0.770534,912.045630,9.212582,13.131313,5.776093,0.0,14.333207,20.0,18.739609,40.0,19.648276,2.564103,"[64, 32]",0.1,0.00010,0.0010,128,250
7,temporal_w30,MLP_tabular_cfg_01,99,16.355485,20.221599,0.770461,872.481172,8.812941,16.161616,7.276713,5.0,15.130634,20.0,21.795317,50.0,18.849738,2.564103,"[64, 32]",0.0,0.00010,0.0010,256,250


## Comparación contra RF y mejor boosting


In [8]:
model_factories, availability_notes = available_tabular_factories(random_state=SEED)
boosting_metrics_path = RESULTS_DIR / "fd001_boosting_metrics.csv"
if boosting_metrics_path.exists():
    previous = pd.read_csv(boosting_metrics_path)
    candidates = previous.loc[previous["model_name"] != "RandomForestRegressor"]
    best_boosting_name = candidates.sort_values(["rmse", "mae"]).iloc[0]["model_name"] if not candidates.empty else "HistGradientBoostingRegressor"
else:
    best_boosting_name = "XGBRegressor" if "XGBRegressor" in model_factories else "HistGradientBoostingRegressor"

best_mlp_name = mlp_metrics.iloc[0]["model_name"]
baseline_predictions = []
for model_name in ["RandomForestRegressor", best_boosting_name]:
    print(f"Entrenando baseline {model_name}")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        pred_df, _ = fit_predict_model(prepared, model_name, model_factories[model_name](), representation)
    baseline_predictions.append(pred_df)

best_mlp_predictions = mlp_predictions.loc[mlp_predictions["model_name"] == best_mlp_name].copy()
all_predictions = pd.concat([best_mlp_predictions, *baseline_predictions], ignore_index=True)
all_metrics = metrics_by_model(all_predictions)
all_bin_metrics = metrics_by_rul_bin(all_predictions)
all_metrics = add_bin_metric_columns(all_metrics, all_bin_metrics)
all_metrics.insert(2, "window_size", DEFAULT_WINDOW_SIZE)
all_metrics.insert(3, "rul_cap", DEFAULT_RUL_CAP)
all_metrics.insert(4, "n_features", len(prepared["feature_columns"]))
all_metrics = all_metrics.merge(config_table, on="model_name", how="left")
all_metrics


Entrenando baseline RandomForestRegressor
Entrenando baseline LGBMRegressor


,representation,model_name,window_size,rul_cap,n_features,n_eval,mae,rmse,r2,cmapss_score,cmapss_score_mean,dangerous_error_pct,mae_rul_0_30,dangerous_error_pct_rul_0_30,mae_rul_30_60,dangerous_error_pct_rul_30_60,mae_rul_60_90,dangerous_error_pct_rul_60_90,mae_rul_90plus,dangerous_error_pct_rul_90plus,hidden_layers,dropout,weight_decay,lr,batch_size,epochs_run
0,temporal_w30,LGBMRegressor,30,125,119,99,12.809901,16.517134,0.846858,397.315013,4.013283,11.111111,4.325694,0.0,8.545874,15.0,16.484272,40.0,17.463165,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,temporal_w30,MLP_tabular_cfg_03,30,125,119,99,12.626827,17.051460,0.836789,449.529187,4.540699,10.101010,4.246185,0.0,8.448703,15.0,15.369181,35.0,17.660883,0.0,"[128, 64, 32]",0.1,0.0001,0.001,256.0,250.0
2,temporal_w30,RandomForestRegressor,30,125,119,99,13.817341,17.912073,0.819898,580.933845,5.868019,14.141414,3.683619,0.0,12.365219,20.0,19.480671,50.0,16.854527,0.0,NaN,NaN,NaN,NaN,NaN,NaN


## Guardado y conclusión


In [9]:
figure_dir = FIGURES_DIR / "fd001_mlp"
figure_dir.mkdir(parents=True, exist_ok=True)

all_metrics.to_csv(RESULTS_DIR / "fd001_mlp_metrics.csv", index=False)
all_predictions.to_csv(RESULTS_DIR / "fd001_mlp_predictions.csv", index=False)
all_bin_metrics.to_csv(RESULTS_DIR / "fd001_mlp_metrics_by_rul_bin.csv", index=False)
plot_validation_diagnostics(all_predictions, figure_dir, "FD001 MLP validation")

best_mlp = all_metrics.loc[all_metrics["model_name"].str.startswith("MLP")].sort_values(["rmse", "mae"]).iloc[0]
best_non_mlp = all_metrics.loc[~all_metrics["model_name"].str.startswith("MLP")].sort_values(["rmse", "mae"]).iloc[0]
if best_mlp["rmse"] < best_non_mlp["rmse"]:
    print("La mejor MLP supera al mejor modelo tabular de comparación por RMSE en esta validación artificial.")
else:
    print("La mejor MLP no supera al mejor modelo tabular de comparación. En FD001, los modelos tabulares con features temporales siguen siendo más robustos en esta validación.")

display(all_metrics.sort_values(["rmse", "mae"]))


La mejor MLP no supera al mejor modelo tabular de comparación. En FD001, los modelos tabulares con features temporales siguen siendo más robustos en esta validación.


,representation,model_name,window_size,rul_cap,n_features,n_eval,mae,rmse,r2,cmapss_score,cmapss_score_mean,dangerous_error_pct,mae_rul_0_30,dangerous_error_pct_rul_0_30,mae_rul_30_60,dangerous_error_pct_rul_30_60,mae_rul_60_90,dangerous_error_pct_rul_60_90,mae_rul_90plus,dangerous_error_pct_rul_90plus,hidden_layers,dropout,weight_decay,lr,batch_size,epochs_run
0,temporal_w30,LGBMRegressor,30,125,119,99,12.809901,16.517134,0.846858,397.315013,4.013283,11.111111,4.325694,0.0,8.545874,15.0,16.484272,40.0,17.463165,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,temporal_w30,MLP_tabular_cfg_03,30,125,119,99,12.626827,17.051460,0.836789,449.529187,4.540699,10.101010,4.246185,0.0,8.448703,15.0,15.369181,35.0,17.660883,0.0,"[128, 64, 32]",0.1,0.0001,0.001,256.0,250.0
2,temporal_w30,RandomForestRegressor,30,125,119,99,13.817341,17.912073,0.819898,580.933845,5.868019,14.141414,3.683619,0.0,12.365219,20.0,19.480671,50.0,16.854527,0.0,NaN,NaN,NaN,NaN,NaN,NaN
